# Análisis Exploratorio — Pokémon Gen I

Exploración del dataset limpio generado por el pipeline ETL.  
Corre primero `python main.py --limit 151` para generar los datos.

**Objetivo:** Entender la distribución de estadísticas, tipos y poder de los 151 Pokémon de la primera generación.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

CSV_PATH = Path('..') / 'data' / 'processed' / 'pokemon_clean.csv'
df = pd.read_csv(CSV_PATH)
print(f'Dataset: {df.shape[0]} filas × {df.shape[1]} columnas')
df.head()

## 1. Resumen estadístico

In [ ]:
stat_cols = ['hp', 'attack', 'defense', 'special_attack', 'special_defense', 'speed', 'bst']
df[stat_cols].describe().round(1)

## 2. Distribución de tipos primarios

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
type_counts = df['type_primary'].value_counts()
bars = ax.bar(type_counts.index, type_counts.values, color='steelblue', edgecolor='white')
ax.bar_label(bars, padding=2, fontsize=9)
ax.set_title('Pokémon por tipo primario — Gen I', fontsize=13, fontweight='bold')
ax.set_xlabel('Tipo')
ax.set_ylabel('Cantidad')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()
print(f'Tipos distintos: {type_counts.shape[0]}  |  Más común: {type_counts.idxmax()} ({type_counts.max()})')

## 3. BST — distribución y top 10

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Histograma BST
ax1.hist(df['bst'], bins=20, color='steelblue', edgecolor='white')
ax1.axvline(df['bst'].mean(), color='tomato', linestyle='--', label=f'Media: {df["bst"].mean():.0f}')
ax1.set_title('Distribución BST', fontweight='bold')
ax1.set_xlabel('Base Stat Total')
ax1.legend()

# Top 10 BST
top10 = df.nlargest(10, 'bst')[['name', 'bst', 'type_primary']].sort_values('bst')
ax2.barh(top10['name'].str.capitalize(), top10['bst'], color='steelblue', edgecolor='white')
ax2.set_title('Top 10 por BST', fontweight='bold')
ax2.set_xlabel('BST')

plt.tight_layout()
plt.show()

## 4. Correlación entre estadísticas

In [ ]:
corr = df[['hp','attack','defense','special_attack','special_defense','speed']].corr()
labels = ['HP','Ataque','Defensa','Sp.Ataque','Sp.Defensa','Velocidad']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=labels, yticklabels=labels, ax=ax, linewidths=0.5)
ax.set_title('Correlación de estadísticas base', fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Distribución por power tier

In [ ]:
tier_order = df.groupby('power_tier')['bst'].mean().sort_values().index.tolist()
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='power_tier', y='bst', order=tier_order, palette='Blues', ax=ax)
ax.set_title('BST por tier de poder', fontweight='bold')
ax.set_xlabel('Tier')
ax.set_ylabel('BST')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

df.groupby('power_tier')['bst'].agg(['count','mean','min','max']).round(1).sort_values('mean')

## 6. Ataque vs Defensa por tipo

In [ ]:
avg = df.groupby('type_primary')[['attack','defense']].mean().round(1)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(avg['attack'], avg['defense'], s=80, color='steelblue', zorder=3)
for tipo, row in avg.iterrows():
    ax.annotate(tipo, (row['attack'], row['defense']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
lim = (avg.min().min() * 0.9, avg.max().max() * 1.05)
ax.plot(lim, lim, 'k--', linewidth=0.8, alpha=0.4, label='Ataque = Defensa')
ax.set_xlabel('Ataque promedio')
ax.set_ylabel('Defensa promedio')
ax.set_title('Ataque vs Defensa promedio por tipo', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Valores nulos — calidad del dataset

In [ ]:
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(1)
quality = pd.DataFrame({'nulos': nulls, 'pct_%': nulls_pct})
print('Columnas con valores nulos:')
print(quality[quality['nulos'] > 0].to_string())
print(f'\nCalidad general: {100 - nulls.sum() / df.size * 100:.1f}% de datos completos')